In [ ]:
import os
import re
import sys
import torch
import random

import numpy as np
import pandas as pd

from datasets import Dataset

from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score
from sentence_transformers import SentenceTransformer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold, train_test_split

from tqdm import tqdm
from tqdm.auto import tqdm
from torch.nn import CrossEntropyLoss
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    set_seed,
    BitsAndBytesConfig
)

def fixar_todas_as_seeds(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

    np.random.seed(seed)

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

    set_seed(seed)

    print(f"Todas as seeds foram fixadas com sucesso para o valor: {seed}")

fixar_todas_as_seeds(42)

Teste de gpu

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

# Tratando amostra do dataset já classificada

Limpeza do Data Frame


In [ ]:
def preprocess_transformer(text):
    text = str(text)
    text = text.strip()
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
df = pd.read_csv("CAMINHO DO CSV")

if 'textClean' not in df.columns:
    df['textClean'] = None

df['commentText'] = df['commentText'].str.replace(r"http\S+|www\S+", " ", regex=True)

df['textClean'] = df['commentText'].apply(preprocess_transformer)

label_map = {"Negativo": 0, "Neutro": 1, "Positivo": 2}
df['feeling'] = df['feeling'].map(label_map)

Divisão e Cross-Validation

In [ ]:

X = df['textClean']
y = df['feeling']

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

splits = list(skf.split(X, y))

Class Weights

In [ ]:
classes = np.unique(y)

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y
)

class_weights = torch.tensor(weights, dtype=torch.float)

print("Class weights:", class_weights)

Weighted Trainer

In [ ]:
class WeightedTrainer(Trainer):
    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):
        labels = inputs.get("labels")

        outputs = model(**inputs)

        logits = outputs.get("logits")

        loss_fct = CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )

        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

# BERTimbau

In [ ]:
bert = "neuralmind/bert-base-portuguese-cased"

bert_tokenizer = AutoTokenizer.from_pretrained(bert)

In [ ]:
!pip install optuna

import optuna

In [ ]:
print("Preparando os dados do Fold 1 para a Busca de Hiperparâmetros...")

tokenizer = AutoTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")

# Pega o primeiro split (Fold 1) dos 1000 comentários
train_idx_search, val_idx_search = next(iter(splits))

X_train_search = X.iloc[train_idx_search].tolist()
y_train_search = y.iloc[train_idx_search].tolist()

X_val_search = X.iloc[val_idx_search].tolist()
y_val_search = y.iloc[val_idx_search].tolist()

train_encodings_search = tokenizer(X_train_search, truncation=True, padding=True, max_length=128)
val_encodings_search = tokenizer(X_val_search, truncation=True, padding=True, max_length=128)

# Classe do Dataset
class ComentariosDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset_search = ComentariosDataset(train_encodings_search, y_train_search)
eval_dataset_search = ComentariosDataset(val_encodings_search, y_val_search)

print("Dados preparados com sucesso!")

In [ ]:
def model_init():
    return AutoModelForSequenceClassification.from_pretrained(
        "neuralmind/bert-base-portuguese-cased",
        num_labels=3
    )

In [ ]:
print("Iniciando a Busca Bayesiana com Optuna...\n")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, predictions, average='macro')
    return {"f1": f1}

training_args_search = TrainingArguments(
    output_dir='/resultados_busca_bertimbau',
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    fp16=True,
    disable_tqdm=True
)

# Instancia o Trainer
trainer_busca = WeightedTrainer(
    model_init=model_init,
    args=training_args_search,
    train_dataset=train_dataset_search,
    eval_dataset=eval_dataset_search,
    compute_metrics=compute_metrics
)

# Define as fronteiras que o Optuna pode testar
def optuna_hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [8, 16]),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 3, 5),
        "weight_decay": trial.suggest_float("weight_decay", 0.01, 0.1),
    }

# Inicia a pesquisa
best_trial = trainer_busca.hyperparameter_search(
    direction="maximize",
    backend="optuna",
    hp_space=optuna_hp_space,
    n_trials=5
)

print("\n🏆 BUSCA CONCLUÍDA! Cole estes hiperparâmetros no loop final:")
print(best_trial.hyperparameters)

In [ ]:
!pip install --upgrade datasets

In [ ]:
f1_scores_bert = []

def tokenize_bert(batch):
    return bert_tokenizer(batch["text"], truncation=True, padding=True, max_length=128)

for train_idx, test_idx in splits:

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    train_dataset = Dataset.from_dict({
        "text": X_train.tolist(),
        "label": y_train.tolist()
    })

    test_dataset = Dataset.from_dict({
        "text": X_test.tolist(),
        "label": y_test.tolist()
    })

    train_dataset = train_dataset.map(tokenize_bert, batched=True)
    test_dataset  = test_dataset.map(tokenize_bert, batched=True)

    train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

    model = AutoModelForSequenceClassification.from_pretrained(bert, num_labels=3).to(device)

    training_args = TrainingArguments(
        output_dir="/results_bert",
        learning_rate=2.9707456131017728e-05,
        per_device_train_batch_size=16,
        num_train_epochs=5,
        weight_decay=0.04011555353637335,
        logging_steps=50,
        save_strategy="no",
        seed=42,
        disable_tqdm=True
    )

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset
    )

    trainer.train()

    preds_output = trainer.predict(test_dataset)
    preds = np.argmax(preds_output.predictions, axis=1)

    f1 = f1_score(y_test, preds, average='macro')
    f1_scores_bert.append(f1)

print("BERTimbau F1 médio:", np.mean(f1_scores_bert))
print("Desvio padrão:", np.std(f1_scores_bert))

# XLM-RoBERTa

In [ ]:
print("Preparando os dados do Fold 1 para a Busca do XLM-RoBERTa...")

roberta_nome = "xlm-roberta-base"
tokenizer_roberta = AutoTokenizer.from_pretrained(roberta_nome)

# Pega o primeiro split (Fold 1) dos 1000 comentários
train_idx_search, val_idx_search = next(iter(splits))

X_train_search = X.iloc[train_idx_search].tolist()
y_train_search = y.iloc[train_idx_search].tolist()

X_val_search = X.iloc[val_idx_search].tolist()
y_val_search = y.iloc[val_idx_search].tolist()

train_encodings_search = tokenizer_roberta(X_train_search, truncation=True, padding=True, max_length=128)
val_encodings_search = tokenizer_roberta(X_val_search, truncation=True, padding=True, max_length=128)

# Classe do Dataset
class ComentariosDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset_search = ComentariosDataset(train_encodings_search, y_train_search)
eval_dataset_search = ComentariosDataset(val_encodings_search, y_val_search)

print("Dados preparados com sucesso!")

In [ ]:
def model_init_roberta():
    return AutoModelForSequenceClassification.from_pretrained(
        "xlm-roberta-base",
        num_labels=3
    )

In [ ]:
print("Iniciando a Busca Bayesiana para o XLM-RoBERTa...\n")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, predictions, average='macro')
    return {"f1": f1}

training_args_search = TrainingArguments(
    output_dir='/resultados_busca_roberta',
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    fp16=True,
    disable_tqdm=True
)

# Instancia o Trainer
trainer_busca = WeightedTrainer(
    model_init=model_init_roberta,
    args=training_args_search,
    train_dataset=train_dataset_search,
    eval_dataset=eval_dataset_search,
    compute_metrics=compute_metrics
)

# Define as fronteiras do que o Optuna pode testar
def optuna_hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [8, 16]),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 3, 5),
        "weight_decay": trial.suggest_float("weight_decay", 0.01, 0.1),
    }

# Inicia a pesquisa
best_trial = trainer_busca.hyperparameter_search(
    direction="maximize",
    backend="optuna",
    hp_space=optuna_hp_space,
    n_trials=5
)

print("\n🏆 BUSCA CONCLUÍDA! Cole estes hiperparâmetros no loop do XLM-RoBERTa:")
print(best_trial.hyperparameters)

In [ ]:
roberta = "xlm-roberta-base"

roberta_tokenizer = AutoTokenizer.from_pretrained(roberta)

In [ ]:
f1_scores_roberta = []

def tokenize_roberta(batch):
    return roberta_tokenizer(batch["text"], truncation=True, padding=True, max_length=128)

for train_idx, test_idx in splits:

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    train_dataset = Dataset.from_dict({
        "text": X_train.tolist(),
        "label": y_train.tolist()
    })

    test_dataset = Dataset.from_dict({
        "text": X_test.tolist(),
        "label": y_test.tolist()
    })

    train_dataset = train_dataset.map(tokenize_roberta, batched=True)
    test_dataset  = test_dataset.map(tokenize_roberta, batched=True)

    if "torchvision" in sys.modules:
        del sys.modules["torchvision"]

    train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

    model = AutoModelForSequenceClassification.from_pretrained(roberta, num_labels=3).to(device)

    training_args = TrainingArguments(
      output_dir="./results_roberta",
      learning_rate=1.951148819758166e-05,
      weight_decay=0.025984269666460565,
      per_device_train_batch_size=16,
      num_train_epochs=5,
      logging_steps=50,
      save_strategy="no",
      seed=42,
      disable_tqdm=True,
      # fp16=True, # Descomentar se estiver usando uma GPU NVIDIA moderna
  )

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset
    )

    trainer.train()

    preds_output = trainer.predict(test_dataset)
    preds = np.argmax(preds_output.predictions, axis=1)

    f1 = f1_score(y_test, preds, average='macro')
    f1_scores_roberta.append(f1)

print("XLM-R F1 médio:", np.mean(f1_scores_roberta))
print("Desvio padrão:", np.std(f1_scores_roberta))

# Llama 3.1

In [ ]:
!pip install --upgrade transformers accelerate bitsandbytes

In [ ]:
from huggingface_hub import login

login("TOKEN HUGGINFACE")

from google.colab import userdata

userdata.get('HF_TOKEN')

In [ ]:
llama = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(llama)
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

# Configurando para 4-bits
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

llama_model = AutoModelForCausalLM.from_pretrained(
    llama,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

In [ ]:
def classify_llama(text):
    prompt = f"""Você é um assistente especializado em analisar o sentimento de comentários do YouTube.
Classifique o sentimento do comentário estritamente como: positivo, negativo ou neutro.

Comentário: "Continuam achando que vao acabar a internet 😂😂😂😂"
Sentimento: neutro

Comentário: "eu não aguento mais ver o Felca em todo lugar!!!😫"
Sentimento: negativo

Comentário: "Apoiado, finalmente uma lei justa pra internet"
Sentimento: positivo

Comentário: "{text}"
Sentimento:"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = llama_model.generate(
        **inputs,
        max_new_tokens=5, # Poucas palavras na resposta
        temperature=0.1,  # Reduz a criatividade para o modelo não viaja no texto
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    # Pegar os tokens gerados
    input_length = inputs["input_ids"].shape[-1]
    generated_ids = outputs[0][input_length:]

    # Decodifica apenas a palavra nova
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().lower()

    if "positivo" in response:
        return 2
    elif "negativo" in response:
        return 0
    else:
        return 1 # Assume neutro se ele gerar algo fora do padrão

In [ ]:

f1_scores_llama = []

print("Iniciando avaliação do LLaMA com Few-Shot...\n")

for fold_idx, (train_idx, test_idx) in enumerate(splits):
    print(f"--- Processando Fold {fold_idx + 1}/5 ---")

    X_test = X.iloc[test_idx]
    y_test = y.iloc[test_idx]

    preds = []

    # Barra de progresso para acompanhar os comentários
    for text in tqdm(X_test, desc=f"Classificando Fold {fold_idx + 1}"):
        pred = classify_llama(text)
        preds.append(pred)

    f1 = f1_score(y_test, preds, average='macro')
    f1_scores_llama.append(f1)
    print(f"F1 do Fold {fold_idx + 1}: {f1:.4f}\n")

print("=== RESULTADOS FINAIS (LLaMA Few-Shot) ===")
print("F1 médio:", np.mean(f1_scores_llama))
print("Desvio padrão:", np.std(f1_scores_llama))

# TabPFN Clássico

In [ ]:
!pip install sentence-transformers tabpfn

from tabpfn import TabPFNClassifier

In [ ]:
print("A carregar o modelo de embeddings semânticos...")
embedding_model = SentenceTransformer('intfloat/multilingual-e5-base')

print("A converter os comentários em vetores (embeddings)...")

# Gera os vetores densos originais
X_embeddings = embedding_model.encode(X.tolist(), show_progress_bar=True)

print(f"Formato original dos embeddings: {X_embeddings.shape} (Pronto para o teste de PCA)")

In [ ]:
os.environ["TABPFN_TOKEN"] = "TOKEN TABPFN"

print("Token do TabPFN configurado com sucesso!")

In [ ]:
componentes_para_testar = [30, 50, 75, 100]
resultados_dimensoes = {}

print("A iniciar o teste rigoroso de dimensões do PCA com TabPFN...\n")

for n_comp in componentes_para_testar:
    print(f">>> A avaliar o cenário com {n_comp} dimensões <<<")

    f1_scores_teste = []

    for fold_idx, (train_idx, test_idx) in enumerate(splits):

        # Separar os embeddings originais (768 dimensões)
        X_train_bruto = X_embeddings[train_idx]
        X_test_bruto = X_embeddings[test_idx]

        y_train_fold = y.iloc[train_idx]
        y_test_fold = y.iloc[test_idx]

        # PCA treina no treino
        pca = PCA(n_components=n_comp, random_state=42)
        X_train_pca = pca.fit_transform(X_train_bruto)

        # PCA aplica a transformação no teste
        X_test_pca = pca.transform(X_test_bruto)

        # TabPFN atua sobre os dados comprimidos isolados
        tabpfn_teste = TabPFNClassifier(device='cuda')
        tabpfn_teste.fit(X_train_pca, y_train_fold)

        preds = tabpfn_teste.predict(X_test_pca)
        f1 = f1_score(y_test_fold, preds, average='macro')
        f1_scores_teste.append(f1)

    f1_medio = np.mean(f1_scores_teste)
    f1_std = np.std(f1_scores_teste)
    resultados_dimensoes[n_comp] = {'media': f1_medio, 'std': f1_std}

    print(f"F1 Médio para {n_comp} dimensões: {f1_medio:.4f} (± {f1_std:.4f})\n")

print("=== RESUMO DOS TESTES DE DIMENSÃO ===")
melhor_dimensao = None
melhor_score = 0

for n_comp, metricas in resultados_dimensoes.items():
    media = metricas['media']
    std = metricas['std']
    print(f"PCA com {n_comp:3d} colunas -> F1-Score: {media:.4f} (± {std:.4f})")

    if media > melhor_score:
        melhor_score = media
        melhor_dimensao = n_comp

print(f"\n🏆 A melhor configuração validada foi com {melhor_dimensao} dimensões!")

# Código do Prenassi adaptado pro Colab

In [ ]:
import gc
import os
import time
import json
import torch
import socket
import random
import copy

import numpy as np
import pandas as pd

from tqdm import tqdm
from peft import LoraConfig
from datetime import datetime
from google.colab import userdata
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig, Trainer, set_seed

MODEL_ID = {
    'Llama3.1-I' : 'meta-llama/Meta-Llama-3.1-8B-Instruct',
    'Llama3.1'   : 'meta-llama/Meta-Llama-3.1-8B'
}

SEED = 2024
set_seed(SEED)

In [ ]:
def get_token():

    token_access = 'SEU TOKEN'
    return token_access

In [ ]:
def transform_json(output_path):
    output_json = {
        "system_prompt": (
            "Você é um assistente especializado em análise de sentimentos. "
            "Sua tarefa é classificar o sentimento do comentário fornecido estritamente "
            "em uma destas três categorias: positivo, neutro ou negativo. "
            "A sua resposta deve conter APENAS o nome da categoria, em letras minúsculas, "
            "sem NENHUM texto adicional, pontuação ou explicação."
        ),
        "categories": ["positivo", "neutro", "negativo"]
    }

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(output_json, f, ensure_ascii=False, indent=4)

def str2bool(x):
    if str(x).lower() in ['y', 'yes', 's', 'sim', '1', 'abacaxi']:
        return True
    return False

def check_if_out_file_exists(args):
    if os.path.exists(args.outfilename):
        raise RuntimeError(f"Erro: O arquivo {args.outfilename} já existe! Mude o nome na configuração ou ative o overwrite.")

def check_if_split_exists(args):
    saida = args.filename+".json"
    if os.path.exists(saida):
        raise RuntimeError(f"Erro: Já existe um output de seleção no caminho {saida}")

def read_dataset(caminho_csv):
    df = pd.read_csv(caminho_csv)

    # Mantém só as colunas que importam
    if 'commentText' in df.columns and 'feeling' in df.columns:
        df = df[['commentText', 'feeling']].copy()
        # Renomeia para o padrão interno do código
        df.rename(columns={'commentText': 'comments', 'feeling': 'label'}, inplace=True)
    else:
        raise ValueError("Erro: O CSV não contém as colunas 'commentText' ou 'feeling'. Verifique o arquivo!")

    df.dropna(subset=['comments', 'label'], inplace=True)
    df.reset_index(drop=True, inplace=True)

    # Padroniza as labels para minúsculo para bater com o LLaMA
    df['label'] = df['label'].str.lower().str.strip()

    return df

def save_file(save_dir, info):
    with open(save_dir, 'w') as arquivo_json:
        json.dump(info, arquivo_json, indent=4)

def print_in_file(msg, filename):
    with open(filename, 'a') as arq:
        arq.write(msg+"\n")

def get_examples(df, prompt_dir, number_of_examples):
    with open(prompt_dir, 'r') as f:
        data = json.load(f)

    categorias = data["categories"]
    texts_for_few_shot = {}

    for categoria in categorias:
        amostras = df[df['label'] == categoria].head(number_of_examples)

        for index, row in amostras.iterrows():
            texts_for_few_shot[index] = {'text': row['comments'], 'label': row['label']}

    print(f"-> Extraídos {len(texts_for_few_shot)} exemplos do CSV para ensinar o modelo (Few-Shot).")
    return texts_for_few_shot

In [ ]:
# Altere os caminhos conforme necessário
class ArgsConfig:
    def __init__(self):
        self.number_of_examples = 3
        self.inputdir = "DIRETORIO BASE"
        self.llm_method = "Llama3.1-I"
        self.overwrite = True
        self.outputdir = "RESULTADOS DO LLAMA"
        self.prompt_dir = "CAMINHO DO PROMPT" # Caminho para o json de prompts que será criado
        self.machine = socket.gethostname()

def args_llm():
    args = ArgsConfig()

    args.outfilename    = f"{args.outputdir}/classification.json"
    args.start_cls_time = datetime.now().strftime("%d-%m-%Y %H:%M:%S")

    print("=== Configurações Carregadas ===")
    for key, value in vars(args).items():
        print(f"{key}: {value}")
    print("================================\n")

    if os.path.exists(args.outfilename) and not args.overwrite:
        print(f"⚠️ Aviso: O arquivo {args.outfilename} já existe e 'overwrite' está Falso.")
        print("Isso pode gerar erros se o código tentar salvar por cima depois.")

    if not os.path.exists(args.outputdir):
        print(f"Criando pasta de saída em: {args.outputdir}")
        os.makedirs(args.outputdir, exist_ok=True)

    # Seed original
    random.seed(1608637542)

    info = {
        "args": vars(args),
        "time_to_classify": [],
        "time_to_classify_avg": [],
        "y_pred_text": [],
    }

    return args, info

# Testar se está funcionando com o comando abaixo
# args, info = args_llm()

In [ ]:
class LLM():
    def __init__(
            self,
            llm_method: str = 'Llama3.1-I',
            prompt_dir = "",
        ):

        self.llm_method = llm_method
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model_name = MODEL_ID[self.llm_method]
        self.prompt_dir = prompt_dir

    def set_model(self, texts_for_few_shot):
        self.get_prompt_(texts_for_few_shot)

        self.token_access = get_token()

        # Comprimindo o LLaMA para 4-bits
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16
        )

        print("Carregando o modelo em 4-bits (Otimizado para a GPU do Colab)...")

        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.float16,
            token=self.token_access,
            trust_remote_code=True,
            use_cache=False
        )

        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_name,
            torch_dtype="auto",
            device_map="auto",
            offload_buffers=True,
            token=self.token_access,
            use_safetensors=True,
            trust_remote_code=True
        )

        self.tokenizer.pad_token = self.tokenizer.eos_token

        self.terminators = [
            self.tokenizer.eos_token_id,
            self.tokenizer.convert_tokens_to_ids("<|eot_id|>")
        ]

    def create_text_prompt(self, predict=False):
        prompt = [
            {
                'role': 'system',
                'content':  self.system_prompt
            }
        ]

        for comment in self.texts_for_few_shot.values():
            prompt += [
                {
                    'role': 'user',
                    'content': f'Input: {comment["text"]}:'
                },
                {
                    'role': 'assistant',
                    'content': f'{comment["label"]}'
                }
            ]

        return prompt

    def get_prompt_(self, texts_for_few_shot):
        with open(self.prompt_dir, 'r') as f:
            data = json.load(f)

        self.system_prompt = data["system_prompt"]
        self.categories = data["categories"]
        self.texts_for_few_shot = texts_for_few_shot
        self.prompt = self.create_text_prompt()
        self.max_new_tokens = max([len(category.split()) for category in self.categories])

    def remove_tokens_for_classification(self, text, total_number_of_tokens, target_number_of_tokens=5000):
        words = text.split()
        tokens_removed = 0
        current_number_of_tokens = total_number_of_tokens

        if current_number_of_tokens > target_number_of_tokens*2:
            step_size = len(words) // 2
        else:
            step_size = len(words) // 20

        while total_number_of_tokens - tokens_removed > target_number_of_tokens:
            words = words[:-step_size]
            truncated_text = " ".join(words)

            prompt = self.create_text_prompt(predict=True)

            inputs = self.tokenizer.apply_chat_template(prompt, add_generation_prompt=True, return_tensors="pt", return_dict=True)

            current_number_of_tokens = inputs['input_ids'][0].numel()
            tokens_removed = total_number_of_tokens - current_number_of_tokens

            if current_number_of_tokens > target_number_of_tokens*2:
                step_size = len(words) // 2
            else:
                step_size = len(words) // 20

        return inputs.to("cuda")

    def add_text_in_prompt_to_classify(self, text):
        prompt = copy.deepcopy(self.prompt)
        prompt += [
            {
                'role': 'user',
                'content': f'{text}'
            }
        ]
        return prompt

    def predict_llm_(self, text):

        default_prompt = self.add_text_in_prompt_to_classify(text)

        inputs = self.tokenizer.apply_chat_template(default_prompt, add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
        if inputs['input_ids'].numel() > 5000:
            print('Removing text', inputs['input_ids'].numel(), end=' ')
            inputs = self.remove_tokens_for_classification(text=text, total_number_of_tokens=inputs['input_ids'].numel())
            print(inputs['input_ids'].numel())

        outputs = self.model.generate(
            inputs['input_ids'],
            attention_mask = inputs['attention_mask'],
            max_new_tokens=self.max_new_tokens+5,
            eos_token_id=self.terminators,
            pad_token_id=self.tokenizer.eos_token_id,
            do_sample=True,
            temperature=0.1,
            top_p=0.9,
            use_cache=False
        )
        response_model = outputs[0][inputs['input_ids'].shape[-1]:]
        response_model = self.tokenizer.decode(response_model, skip_special_tokens=True)
        response_model = response_model.lower()

        return response_model

    def predict(self, data):
        print(self.llm_method)

        y_text = []
        X = data['comments'].tolist()

        for index, text in enumerate(tqdm(X, desc="Predict", ascii=True)):

            if index in self.texts_for_few_shot.keys():
                y_text.append(self.texts_for_few_shot[index]['label'])
                continue

            response_model = self.predict_llm_(f'Input: {text}')

            while response_model not in self.categories:
                print('Regenerating response.')
                new_text = f'Attention! Classify only into the categories you were instructed to.\nInput: {text}'
                response_model = self.predict_llm_(new_text)

            y_text.append(response_model)
            torch.cuda.empty_cache()

        return y_text

In [ ]:
!pip install -U bitsandbytes accelerate transformers

In [ ]:
def run_classification(number_of_examples=3):
    print(f"--- Iniciando Classificação Few-Shot com {number_of_examples} exemplos por classe ---")

    base_dir = "DIRETORIO BASE"

    prompt_dir = f"{base_dir}/prompt.json"
    csv_path = f"{base_dir}/classified_comments.csv"
    outfilename = f"{base_dir}/resultados_llama/sentiment_analysis_fewshot.json"

    llm_method = 'Llama3.1-I'
    seed = 2024

    os.makedirs(os.path.dirname(outfilename), exist_ok=True)
    info = {}

    print("\n1. Preparando o Prompt de Análise de Sentimentos...")

    # Cria o JSON de regras no caminho especificado
    transform_json(prompt_dir)

    print("\n2. Lendo o Dataset...")
    df = read_dataset(csv_path)
    print(f"Dataset carregado com {len(df)} linhas válidas.")

    print("\n3. Extraindo exemplos para o Few-Shot...")

    texts_for_few_shot = get_examples(df, prompt_dir, number_of_examples)

    print("\n4. Inicializando e baixando o LLaMA (isso pode demorar um pouco)...")
    llm = LLM(llm_method=llm_method, prompt_dir=prompt_dir)
    llm.set_model(texts_for_few_shot)

    print("\n5. Iniciando predições (Predict!)...")
    classification_start_time = time.time()

    y_pred_text = llm.predict(df)

    classification_end_time = time.time()

    print("\n6. Salvando resultados...")
    info["time_to_classify"] = classification_end_time - classification_start_time
    info["time_to_classify_avg"] = (classification_end_time - classification_start_time) / len(df)
    info["y_pred_text"] = y_pred_text
    info["prompt"] = llm.system_prompt
    info["seed"] = seed

    save_file(outfilename, info)
    print(f"✅ Classificação concluída! Resultados salvos em: {outfilename}")

    # Limpeza da memória da placa de vídeo
    del llm
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("🧹 Memória da GPU liberada com sucesso.")

run_classification(number_of_examples=3)

In [ ]:
from sklearn.metrics import f1_score, classification_report

print("--- Calculando o F1-Score do LLaMA 3.1 ---")

base_dir = "DIRETORIO BASE"
csv_path = f"{base_dir}/classified_comments.csv"
json_path = f"{base_dir}/resultados_llama/sentiment_analysis_fewshot.json"

df = pd.read_csv(csv_path)
df.dropna(subset=['commentText', 'feeling'], inplace=True)

y_true = df['feeling'].str.lower().str.strip().tolist()

with open(json_path, 'r', encoding='utf-8') as f:
    resultados = json.load(f)

y_pred = resultados['y_pred_text']

y_pred = [str(pred).lower().strip() for pred in y_pred]

if len(y_true) != len(y_pred):
    print(f"⚠️ Atenção: O tamanho do gabarito ({len(y_true)}) está diferente das predições ({len(y_pred)}).")
else:
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)

    print(f"\nF1-Score Macro: {f1_macro:.4f}")
    print(f"F1-Score Micro: {f1_micro:.4f}\n")

    print("=== Relatório Detalhado por Classe ===")
    relatorio = classification_report(y_true, y_pred, zero_division=0)
    print(relatorio)